In [1]:

# transfer_learning_finetuning_pytorch.ipynb

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import os

In [2]:

# ✅ Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cuda


In [3]:
# ✅ Data transforms (like resizing and normalization)
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                         [0.229, 0.224, 0.225])  # ImageNet std
])


In [4]:

# ✅ Dataset loading (replace with your own path)
train_dir = "dogs_vs_cats/train"
val_dir = "dogs_vs_cats/test"

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [5]:
# ✅ Load pretrained VGG16
vgg16 = models.vgg16(pretrained=True)

d:\Programming\Deep_Learning_Concept\.deeplr\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Programming\Deep_Learning_Concept\.deeplr\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
# ✅ Freeze all layers except block5_conv1 onwards
for name, param in vgg16.features.named_parameters():
    if '24' in name:  # 'block5_conv1' equivalent
        break
    param.requires_grad = False

# ✅ Replace classifier
vgg16.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(25088, 256),
    nn.ReLU(),
    nn.Linear(256, 1),
    nn.Sigmoid()  # For binary classification
)

vgg16 = vgg16.to(device)

In [7]:

# ✅ Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, vgg16.parameters()), lr=0.0001)

In [8]:
# ✅ Training loop (simplified)
for epoch in range(5):  # You can increase epochs
    vgg16.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 44.5258
Epoch 2, Loss: 19.2174
Epoch 3, Loss: 12.8786
Epoch 4, Loss: 10.4084
Epoch 5, Loss: 10.6085
